# T5Gemma S-S: reconstruct entities masked in summaries (2K context)

This notebook fine-tunes `google/t5gemma-s-s-ul2` on the reversed version of the earlier entity-reconstruction task:

- the encoder sees the **unaltered original document** plus its **entity-masked summary**;
- the preparation script uses spaCy `en_core_web_trf` to detect and mask named entities independently in both the original document and summary;
- summary entities are replaced from left to right with T5Gemma's atomic placeholder tokens (`<unused0>`, `<unused1>`, ...);
- the decoder target is `<unused0> entity 0 <unused1> entity 1 ...`.

The standalone `prepare_summary_entity_dataset.py` script produces a model-independent dataset with literal `[mask]` placeholders and aligned entity/type lists. This notebook converts those masks to T5Gemma sentinels, constructs targets, and applies model-specific token-length filtering in memory.

In [ ]:
# Run these once in a fresh environment, then restart the kernel if requested.
# %pip install torch transformers datasets sentencepiece

import json
import math
import random
import re
import time
from pathlib import Path

import torch
from datasets import concatenate_datasets, load_from_disk
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    get_linear_schedule_with_warmup,
)

In [ ]:
# Optional for gated Hugging Face models:
# from huggingface_hub import login
# login()

In [ ]:
PREPARED_DATA_PATHS = {
    "cnn_dailymail": "data/summary_entities_masked/cnn_dailymail",
    "xsum": "data/summary_entities_masked/xsum",
    "samsum": "data/summary_entities_masked/samsum",
    "multi_news": "data/summary_entities_masked/multi_news",
    "billsum": "data/summary_entities_masked/billsum",
}
MODEL_NAME = "google/t5gemma-s-s-ul2"
RUN_DIR = Path("models/t5gemma-s-s-ul2-summary-entities-2k")

# The source must fit without truncation. 2K means 2,048 tokenizer input IDs.
MAX_INPUT_LENGTH = 2048
MAX_SENTINELS = 100

BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 16
EPOCHS = 1
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.02
MAX_GRAD_NORM = 1.0

LOG_EVERY_STEPS = 100
EVAL_EVERY_STEPS = 500
SAVE_EVERY_STEPS = 500

SEED = 42
USE_AMP = True

# Set to a checkpoint directory from this run to restore weights and training state.
RESUME_FROM = None

RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = RUN_DIR / "loss_log.jsonl"

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_enabled = USE_AMP and device.type == "cuda"
if amp_enabled and torch.cuda.is_bf16_supported():
    amp_dtype = torch.bfloat16
elif amp_enabled:
    amp_dtype = torch.float16
else:
    amp_dtype = torch.float32
scaler_enabled = amp_enabled and amp_dtype == torch.float16

print(
    f"device={device}, mixed_precision={amp_enabled}, "
    f"dtype={amp_dtype}, grad_scaler={scaler_enabled}"
)

tokenizer_source = RESUME_FROM or MODEL_NAME
model_source = RESUME_FROM or MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_source,
    torch_dtype=amp_dtype,
).to(device)
model.gradient_checkpointing_enable()
model.config.use_cache = False

sentinel_tokens = [f"<unused{i}>" for i in range(MAX_SENTINELS)]
sentinel_ids = tokenizer.convert_tokens_to_ids(sentinel_tokens)
invalid_sentinels = [
    token
    for token, token_id in zip(sentinel_tokens, sentinel_ids)
    if token_id is None or token_id == tokenizer.unk_token_id
]
if invalid_sentinels or len(set(sentinel_ids)) != MAX_SENTINELS:
    raise ValueError(
        "The tokenizer must contain 100 distinct T5Gemma <unusedN> tokens; "
        f"missing/invalid: {invalid_sentinels}"
    )

print(f"Validated {len(sentinel_ids)} numbered T5 sentinels")

## Load the prepared dataset

Run `python prepare_summary_entity_dataset.py` from the repository root first. It creates a separate DatasetDict for each corpus while preserving every original split (`train`, `validation`, `test`, and `ca_test` where present). Every row contains the original document and summary, both masked forms, and aligned entity/type lists for both sides.

In [ ]:
required_splits = {"train"}
required_columns = {
    "original_text", "summary", "masked_text", "masked_summary",
    "masked_text_entities", "masked_summary_entities",
    "masked_text_entity_types", "masked_summary_entity_types",
}
prepared_datasets = {}
for dataset_name, dataset_path in PREPARED_DATA_PATHS.items():
    prepared = load_from_disk(dataset_path)
    missing_splits = required_splits - set(prepared.keys())
    if missing_splits:
        raise ValueError(
            f"{dataset_name} is missing splits: {missing_splits}"
        )
    for split_name in prepared.keys():
        missing = required_columns - set(prepared[split_name].column_names)
        if missing:
            raise ValueError(
                f"{dataset_name}/{split_name} is missing columns: {missing}"
            )
    prepared_datasets[dataset_name] = prepared

MASK_PATTERN = re.compile(r"\[mask\]")


def make_source(original_text, numbered_summary):
    return (
        f"document:\n{original_text}\n\n"
        f"masked summary:\n{numbered_summary}"
    )


def number_masks(masked_summary, entities):
    mask_count = len(MASK_PATTERN.findall(masked_summary))
    if mask_count != len(entities):
        raise ValueError(f"Found {mask_count} masks but {len(entities)} entities")
    if mask_count == 0 or mask_count > MAX_SENTINELS:
        raise ValueError(f"Unsupported mask count: {mask_count}")

    index = 0
    def replace_mask(_match):
        nonlocal index
        token = f"<unused{index}>"
        index += 1
        return token

    numbered_summary = MASK_PATTERN.sub(replace_mask, masked_summary)
    target = " ".join(
        f"<unused{index}> {entity}"
        for index, entity in enumerate(entities)
    )
    return numbered_summary, target


def add_training_fields(batch):
    sources = []
    targets = []
    objective_valid = []
    for original_text, masked_summary, entities, entity_types in zip(
        batch["original_text"], batch["masked_summary"],
        batch["masked_summary_entities"],
        batch["masked_summary_entity_types"],
    ):
        try:
            if len(entities) != len(entity_types):
                raise ValueError("Entity and entity-type lists are misaligned")
            numbered_summary, target = number_masks(masked_summary, entities)
            source = make_source(original_text, numbered_summary)
            valid = True
        except (TypeError, ValueError):
            source, target, valid = "", "", False
        sources.append(source)
        targets.append(target)
        objective_valid.append(valid)

    source_ids = tokenizer(
        sources, truncation=True, max_length=MAX_INPUT_LENGTH + 1
    )["input_ids"]
    source_lengths = [len(ids) for ids in source_ids]
    return {
        "source": sources,
        "target": targets,
        "source_length": source_lengths,
        "sequence_valid": [
            valid
            and source_length <= MAX_INPUT_LENGTH
            for valid, source_length in zip(objective_valid, source_lengths)
        ],
    }


def prepare_for_training(split, split_name):
    split = split.map(
        add_training_fields,
        batched=True,
        batch_size=256,
        desc=f"Building T5Gemma objective: {split_name}",
    )
    split = split.filter(
        lambda valid: valid,
        input_columns=["sequence_valid"],
        desc=f"Length-valid rows: {split_name}",
    )
    return split.remove_columns(["sequence_valid"])


train_raw = concatenate_datasets([
    prepare_for_training(dataset["train"], f"{name}/train")
    for name, dataset in prepared_datasets.items()
]).shuffle(seed=SEED)
eval_raw = concatenate_datasets([
    prepare_for_training(
        dataset["validation" if "validation" in dataset else "test"],
        f"{name}/{'validation' if 'validation' in dataset else 'test'}",
    )
    for name, dataset in prepared_datasets.items()
]).shuffle(seed=SEED)
print(f"training-ready train={len(train_raw):,}, eval={len(eval_raw):,}")

In [ ]:
for row in train_raw.select(range(min(3, len(train_raw)))):
    print("SOURCE:\n", row["source"])
    print("TARGET:\n", row["target"])
    print("MASKED TEXT:\n", row["masked_text"])
    print("TEXT ENTITIES:\n", row["masked_text_entities"])
    print("TEXT ENTITY TYPES:\n", row["masked_text_entity_types"])
    print("SUMMARY ENTITIES:\n", row["masked_summary_entities"])
    print("SUMMARY ENTITY TYPES:\n", row["masked_summary_entity_types"])
    print("-" * 100)

In [ ]:
def tokenize_row(row):
    encoded = tokenizer(
        row["source"],
        max_length=MAX_INPUT_LENGTH,
        truncation=False,
    )
    target_tokens = tokenizer(
        text_target=row["target"],
        truncation=False,
    )
    encoded["labels"] = target_tokens["input_ids"]
    return encoded


train_set = train_raw.map(
    tokenize_row,
    remove_columns=train_raw.column_names,
    desc="Tokenizing train data",
)
eval_set = eval_raw.map(
    tokenize_row,
    remove_columns=eval_raw.column_names,
    desc="Tokenizing eval data",
)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
)
train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    pin_memory=device.type == "cuda",
)
eval_loader = DataLoader(
    eval_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    pin_memory=device.type == "cuda",
)

diagnostic_batch = next(iter(train_loader))
input_ids = diagnostic_batch["input_ids"]
labels = diagnostic_batch["labels"]
valid_labels = labels[labels != -100]

assert input_ids.min().item() >= 0
assert valid_labels.numel() > 0 and valid_labels.min().item() >= 0
assert input_ids.shape[1] <= MAX_INPUT_LENGTH
assert len(tokenizer) <= model.get_input_embeddings().num_embeddings
assert max(sentinel_ids) < model.get_input_embeddings().num_embeddings

model.eval()
with torch.no_grad(), torch.autocast(
    device_type=device.type,
    enabled=amp_enabled,
    dtype=amp_dtype,
):
    diagnostic_loss = model(
        **{key: value.to(device) for key, value in diagnostic_batch.items()}
    ).loss
model.train()

if not torch.isfinite(diagnostic_loss):
    raise FloatingPointError(f"Initial forward loss is {diagnostic_loss.item()}")
print(
    f"diagnostic_loss={diagnostic_loss.item():.4f}, "
    f"input_shape={tuple(input_ids.shape)}, labels={valid_labels.numel()}"
)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
updates_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION)
total_updates = updates_per_epoch * EPOCHS
warmup_steps = int(total_updates * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_updates,
)
scaler = torch.cuda.amp.GradScaler(enabled=scaler_enabled)
global_step = 0
start_epoch = 0

if RESUME_FROM:
    state_path = Path(RESUME_FROM) / "training_state.pt"
    if state_path.exists():
        state = torch.load(state_path, map_location=device)
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        if state.get("scaler") is not None:
            scaler.load_state_dict(state["scaler"])
        global_step = state["global_step"]
        start_epoch = state["epoch"]
        print(f"Resumed from epoch={start_epoch}, step={global_step}")

print(
    f"updates_per_epoch={updates_per_epoch:,}, "
    f"total_updates={total_updates:,}, warmup={warmup_steps:,}"
)

In [ ]:
def append_log(record):
    record = {"time": time.time(), **record}
    with LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record) + "\n")


@torch.no_grad()
def evaluate():
    model.eval()
    weighted_loss = 0.0
    target_tokens = 0
    for batch in eval_loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        count = (batch["labels"] != -100).sum().item()
        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=amp_enabled,
        ):
            loss = model(**batch).loss
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite validation loss")
        weighted_loss += loss.item() * count
        target_tokens += count
    model.train()
    return weighted_loss / max(target_tokens, 1)


def save_checkpoint(epoch, step):
    checkpoint = RUN_DIR / f"checkpoint-step-{step}"
    checkpoint.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(checkpoint)
    tokenizer.save_pretrained(checkpoint)
    torch.save(
        {
            "epoch": epoch,
            "global_step": step,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict() if scaler_enabled else None,
            "config": {
                "model": MODEL_NAME,
                "prepared_data_paths": PREPARED_DATA_PATHS,
                "objective": "original document + entity-masked summary -> entities",
                "max_input_length": MAX_INPUT_LENGTH,
            },
        },
        checkpoint / "training_state.pt",
    )
    print(f"saved checkpoint: {checkpoint}")

In [ ]:
optimizer.zero_grad(set_to_none=True)
interval_loss = 0.0
interval_microbatches = 0

for epoch in range(start_epoch, EPOCHS):
    model.train()
    for micro_step, batch in enumerate(train_loader, start=1):
        batch = {key: value.to(device) for key, value in batch.items()}
        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype,
            enabled=amp_enabled,
        ):
            raw_loss = model(**batch).loss
            loss = raw_loss / GRADIENT_ACCUMULATION

        if not torch.isfinite(raw_loss):
            raise FloatingPointError(
                f"Non-finite loss at epoch={epoch}, micro_step={micro_step}, "
                f"input_shape={tuple(batch['input_ids'].shape)}, dtype={amp_dtype}"
            )

        scaler.scale(loss).backward()
        interval_loss += raw_loss.item()
        interval_microbatches += 1

        should_update = (
            micro_step % GRADIENT_ACCUMULATION == 0
            or micro_step == len(train_loader)
        )
        if not should_update:
            continue

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

        if global_step % LOG_EVERY_STEPS == 0:
            record = {
                "split": "train",
                "epoch": epoch,
                "step": global_step,
                "loss": interval_loss / max(interval_microbatches, 1),
                "learning_rate": scheduler.get_last_lr()[0],
            }
            append_log(record)
            print(record)
            interval_loss = 0.0
            interval_microbatches = 0

        if global_step % EVAL_EVERY_STEPS == 0:
            eval_loss = evaluate()
            record = {
                "split": "eval",
                "epoch": epoch,
                "step": global_step,
                "loss": eval_loss,
                "perplexity": math.exp(min(eval_loss, 20)),
            }
            append_log(record)
            print(record)

        if global_step % SAVE_EVERY_STEPS == 0:
            save_checkpoint(epoch, global_step)

    eval_loss = evaluate()
    record = {
        "split": "eval_epoch",
        "epoch": epoch,
        "step": global_step,
        "loss": eval_loss,
        "perplexity": math.exp(min(eval_loss, 20)),
    }
    append_log(record)
    print(record)
    save_checkpoint(epoch + 1, global_step)

In [ ]:
FINAL_DIR = RUN_DIR / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"Final model saved to {FINAL_DIR.resolve()}")
print(f"Loss log saved to {LOG_PATH.resolve()}")

In [ ]:
if LOG_PATH.exists():
    loss_history = [
        json.loads(line)
        for line in LOG_PATH.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    print(loss_history[-10:])
else:
    print("No loss log exists yet.")